# IOAI — 2026 Regional 9 10 Asteroids (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train_data.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-asteroids.zip', '/tmp/ast.zip')
    with zipfile.ZipFile('/tmp/ast.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/train_data.csv' if os.path.exists('data/train_data.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

In [ ]:
root_path = "data"
seed = 333

# !cat {root_path}/starter_kit.py
!ls {root_path}

# Cleaning

In [ ]:
!head -n 15 {root_path}/train_data.csv

In [ ]:
pd.read_csv(f"{root_path}/train_data.csv")

In [ ]:
with open(f"{root_path}/train_data.csv", "r") as f:
    lines = f.readlines()

    good_lines = [l for l in lines if l.count(',')]
    good_data = ''.join(good_lines)

with open(f"{root_path}/train_data_cleaned.csv", "w") as f:
    f.write(good_data)

In [ ]:
df = pd.read_csv(f"{root_path}/train_data_cleaned.csv")
df.head()

# Subtask 1

In [ ]:
subtask1 = df["Class"]

# Subtask 2

## Data

In [ ]:
df_test = pd.read_csv(f"{root_path}/test_data.csv")

In [ ]:
plt.figure(figsize=(12 // 2, 4 // 2))
sns.scatterplot(x="X", y="Y", data=df, hue="Class")
plt.show()

plt.figure(figsize=(12 // 2, 4 // 2))
sns.scatterplot(x="Z", y="X", data=df, hue="Class")
plt.show()

In [ ]:
def prep_df(df: pd.DataFrame):
    df = df.drop(["datapointID", "Class"], errors='ignore', axis=1)
    return df

In [ ]:
df_train = prep_df(df)

plt.figure(figsize=(9,3))
sns.histplot(df["Class"])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df_train, df["Class"], test_size=0.2, stratify=df["Class"], random_state=seed)

## Model

In [ ]:
def evaluate(clf):
    cv = cross_val_score(clf, X_train, y_train, cv=3, scoring='f1', n_jobs=-1)
    cv = cv.mean() - cv.std()

    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    return cv.item(), f1_score(y_test, preds, pos_label=1)

In [ ]:
lr = LogisticRegression()

evaluate(lr)

In [ ]:
svc = SVC(class_weight="balanced", random_state=seed)

evaluate(svc)

In [ ]:
svc = SVC(gamma=100, class_weight="balanced", random_state=seed)

evaluate(svc)

In [ ]:
test_feats = prep_df(df_test)
subtask2 = svc.predict(test_feats)

# Submission

In [ ]:
def build_sub(sid, ans):
    return pd.DataFrame({
        "datapointID": (df_test if sid == 2 else df)["datapointID"],
        "subtaskID": sid,
        "answer": ans
    })

subtasks = [
    (1, subtask1),
    (2, subtask2)
]

submission = pd.concat([build_sub(sid, ans) for sid, ans in subtasks])

In [ ]:
submission.head()

In [ ]:
submission.to_csv(f"{root_path}/submission.csv", index=False)

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)